# レッスン18: 暗号化レシートでAIエージェントを保護する

## ハンズオンノートブック

このノートブックでは4つの演習を通して学びます：

1. エージェントツールコールのための<strong>最初のレシートに署名し</strong>、検証する。
2. <strong>レシートを改ざんして</strong>検証が失敗する様子を見る。
3. **3つのレシートチェーンを構築し**、チェーンの完全性を確認する。
4. **Microsoft Agent Frameworkツールコールをラップし**、すべてのアクションがレシートを発行するようにする。

すべての暗号化プリミティブは良く管理されたライブラリ（Ed25519用の`pynacl`、RFC 8785準拠のカノニカルJSON用の`jcs`、Python標準ライブラリのSHA-256用`hashlib`）からインポートしています。レシートのロジック自体は純粋なPythonなので読み書きが可能です。

セルは順番に実行してください。各セクションは短く独立しています。


## セットアップ

2つの依存関係をインストールします。どちらも許容的なライセンス（Apache-2.0 / MIT）です。


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## ヘルパーユーティリティ

これらの2つのヘルパーは、base64urlエンコード（パディングなし）と任意のオブジェクトのSHA-256ハッシュ化を処理します。これにより、ノートブックの残りの部分はレシートのロジック自体に集中できます。


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## セクション 1: 最初のレシートに署名する

Contoso Travel のエージェントがシドニーからロサンゼルス行きのフライトを顧客のために調べたと想像してください。このツールの呼び出しを署名済みレシートとして記録し、将来の監査者が私たちを信用せずに検証できるようにしたいと思います。

### ステップ 1.1: 署名用キーを生成する

本番環境では、エージェントの署名用キーはハードウェアセキュリティモジュール（HSM）や Azure Key Vault、または同様の保護されたストアに保存されます。このレッスンではメモリ上に新しいキーを生成します。


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### ステップ 1.2: レシートペイロードの構築

ペイロードには、レシートが証明すべきすべての情報が含まれます：誰が行動を起こしたか、どのツールを使ったか、どんな引数を使ったか、何が返ってきたか、どのポリシーの下で、そしていつか。引数や結果はレシートに機密情報が漏れないように、インラインで含めるのではなくハッシュ化します。


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### ステップ 1.3: 署名してレシートを組み立てる

3つのステップ：

1. JCS を使用してペイロードを規範化し、同じ論理的レシートを生成する2つの実装がバイト単位で同一のバイトを生成するようにする。
2. その規範化されたバイト列を SHA-256 でハッシュ化する。
3. Ed25519 の秘密鍵でハッシュに署名する。

署名は元のペイロードに添付され、最終的なレシートが生成される。


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### ステップ 1.4: 受領書の検証

検証はプロセスを逆に行います。署名を取り除き、正規ハッシュを再計算し、受領書の公開鍵に対して署名をチェックします。

この検証を行う監査人は、受領書自体以外に私たちから何も必要としません。呼び出すサービスも、照会するキー・ディレクトリも、信頼も必要ありません。


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

`Receipt is valid: True` と表示されるはずです。エージェントは最初の暗号的に署名された監査記録を生成しました。


## セクション 2: レシートの改ざん

レシートの最大の特徴は改ざんが一目で分かることです。これを証明しましょう。

レシートの文字を正確に1文字だけ変更し、検証が失敗する様子を見てみます。


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### 何が起こったのですか？

`policy_id` を変更したとき、正規化されたバイト列が変わりました。そのバイト列の SHA-256 ハッシュも変わりました。署名（元のハッシュに対して行われたもの）は新しいハッシュと一致しなくなりました。検証は正しく `False` を返しました。

攻撃者が秘密鍵を持っていない限り、領収書のいかなるフィールドを変更しても検証を通すことはできません。秘密鍵がキーコンテナーにあり、公開鍵が公開されている限り、改ざんを隠すことは不可能です。

ご自分で試してみてください。上のセルで `tool_name` や `agent_id`、`timestamp` を変更して再実行してください。どの変更も無効な領収書を生成します。


## セクション 3: レシートを連結する

1つのレシートは1つのアクションを保護します。ほとんどのエージェントは多くのアクションを実行します。連続した全体のシーケンスを改ざん検知可能にするために、各レシートのペイロードに前のレシートのハッシュを含めて、各レシートを前のレシートと連結します。

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

もし誰かがレシートを削除または順序を入れ替えた場合、そのポイントでチェーンは正確に切れます。その後の任意のレシートの検証は失敗します。なぜなら、`previous_receipt_hash` がもはやその前任者の実際のハッシュと一致しないためです。


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

次に、真ん中のレシートを改ざんしてチェーンを切り、再度検証してください。改ざんされたレシートは署名チェックに失敗し、さらに次のレシートもチェーンリンクチェックに失敗します（`previous_receipt_hash` が修正された真ん中のレシートのハッシュと一致しなくなるためです）。


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

レシート0はまだ検証に成功します（変更されておらず、依存する前のレシートがないため）。レシート1は `tool_args_hash` を変更したため、署名検証に失敗します。レシート2は、元の（現在変更された）レシート1に対して計算された `previous_receipt_hash` によって連鎖のリンク検証に失敗します。

攻撃者が変更されたレシート1を再度署名したとしても（これは秘密鍵なしでは不可能ですが）、レシート2の連鎖リンク不一致により改ざんが明らかになります。変更を隠すためには、攻撃者は変更点以降のすべてのレシートを再署名する必要があり、それには秘密鍵の所有が必要です。


## セクション 4: レシート署名でエージェントツール呼び出しをラップする

実際の運用では、すべてのエージェント作者が `make_receipt` を呼び出すのを覚えている必要はありません。すべてのツール呼び出しでレシート署名が自動的に行われるようにしたいのです。

ここに最もシンプルなパターンがあります。任意の呼び出し可能なツール関数を受け取り、それをレシートを発行するバージョンに変換するラッパークラスです。これは Microsoft Agent Framework (`agent_framework.foundry`) を含むあらゆるエージェントフレームワークに適応します。

Microsoft Foundryプロジェクトがセットアップされていなくても、以下のローカルモックでパターンを示しています。


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


### Microsoft Agent Frameworkとの統合

上記の`ReceiptedTool`ラッパーはフレームワークに依存しません。Microsoft Agent Frameworkで構築されたエージェント内で使用するには、ラップされた関数をツールとして登録します。スケッチ（モックを実際のMicrosoft Foundryツール登録に置き換えます）：

```python
# 統合形状を示す擬似コード。
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="あなたはContoso Travelのエージェントです...",
#     tools=[receipted_lookup],   # ラップされたツール、元の関数ではありません
# )
# response = agent.run("シドニーからロサンゼルスへの6月のフライトを探してください。")
#
# # 実行後、エージェントが呼び出したすべてのツールコールには署名済みのレシートがあります:
# audit_chain = receipted_lookup.receipts
```

エージェントフレームワークはレシートについて何も知る必要はありません。レシート署名はツールの周囲にラップされており、フレームワークに組み込まれてはいません。これは、エージェントを再構築せずに既存のエージェントコードに出所情報を追加する方法です。


## 振り返りと発展課題

これまでに行ったこと:

- Ed25519 キーペアを生成した。
- エージェントツール呼び出しのためのレシートを作成し、署名した。
- 公開鍵だけを使ってレシートの検証をオフラインで行った。
- レシートを改ざんし、検証が失敗することを確認した。
- ハッシュで連鎖した3つのレシートのシーケンスを作成した。
- 連鎖の途中を改ざんし、署名失敗と連鎖リンク失敗の両方を確認した。
- ツール関数を自動レシート署名でラップした。

**発展課題。** レシートスキーマに `request_id` フィールド（分散トレーシング用の UUID）を追加してください。`make_receipt` を更新して含め、レシートがエンドツーエンドで検証されることを確認します。その後、署名後にフィールドを変更し、検証が失敗することを確認してください。これにより、標準的なエンコーディングのすべてのバイトが署名にどのように寄与するかを内面化できます。

**重要な境界。** レシートは3つのことだけを証明します: 帰属（この鍵がこの内容に署名した）、完全性（署名後に内容が変更されていない）、並び順（このレシートはあのレシートより後に発行された）。レシートはエージェントの行動が正しかったこと、`policy_id` で指定されたポリシーが実際に評価されたこと、またはエージェントがすべてのルールに従ったことを証明しません。レシートは基盤です。ガバナンスはその上に構築するシステムです。

その境界を念頭に置いて、レッスンのREADMEをもう一度読み返しましょう。チームがレシートについて犯しがちな最も一般的な誤りは「レシートがある＝管理されている」と思い込むことです。そうではありません。レシートはエージェントの振る舞いを監査可能にしますが、それを正しいとはしません。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：
本書類は AI 翻訳サービス [Co-op Translator](https://github.com/Azure/co-op-translator) を使用して翻訳されています。正確性を期していますが、自動翻訳には誤りや不正確な部分が含まれる可能性があることをご承知おきください。原文の原語版が正式な情報源とみなされるべきです。重要な情報については、専門の人間による翻訳を推奨します。本翻訳の利用により生じたいかなる誤解や解釈違いについても、当方は責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
